<a href="https://colab.research.google.com/github/osadchayakarina782-wq/nlp-hw-osadchaya/blob/main/%D0%9B%D0%B0%D0%B1_2_%22w2v_hw_ipynb%22.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

В этом практикуме мы рассмотрим работу с библиотекой **Gensim** для работы с векторными представлениями текста

Мы рассмотрим
- **Word2Vec** - векторные представления слов
- **FastText** - улучшенные представления с учетом морфологии  
- **Doc2Vec** - векторные представления документов


In [ ]:
!pip install gensim

import gensim
import gensim.downloader as api
from gensim.models import Word2Vec, FastText, Doc2Vec
from gensim.models.doc2vec import TaggedDocument
import numpy as np

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 62.7 MB/s eta 0:00:00


## Часть 1: Word2Vec

### Что такое Word2Vec?

Word2Vec преобразует слова в векторы чисел так, что семантически похожие слова оказываются близко в векторном пространстве.

**Два основных алгоритма:**
- **CBOW** - предсказывает слово по контексту
- **Skip-gram** - предсказывает контекст по слову

**Загрузка предобученной модели**

In [ ]:
w2v_model = api.load('glove-wiki-gigaword-100')

print(f"Размер словаря: {len(w2v_model.key_to_index)}")
print(f"Размерность векторов: {w2v_model.vector_size}")

[==================================================] 100.0% 128.1/128.1MB downloaded
Размер словаря: 400000
Размерность векторов: 100


Найдите документацию `gensim`: какие датасеты кроме `glove-wiki-gigaword-100` доступны в библиотеке?

Выберите 3 датасета и кратко опишите их (источник данных, примерный объем, зачем такой датасет может использоваться)

Помимо glove-wiki-gigaword-100, существует еще немало датасетов. Рассмотрим некоторые из них:

1) wiki-news-300d-1M-subword.vec обучена на объединенном корпусе Wikipedia 2017, UMBC webbase corpus и наборе новостей с statmt.org. Это модель fastText для английского языка, содержащая 1 миллион векторов слов.

2. word2vec-ruscorpora-300 основана на материалах НКРЯ, в модели содержится ~185 тысяч векторов. Актуальна при работе с русскоязычными текстами.

3. crawl-300d-2M-subword.zip обучена на огромном корпусе Common Crawl, который содержит 600 миллиардов токенов. В результате получилось 2 миллиона векторных представлений слов.

**Базовые операции с векторами**

In [ ]:
# Получаем вектор слова
vector = w2v_model['computer']
print(f"Вектор слова 'computer': {vector[:5]}...")  # Показываем первые 5 чисел

# Вычисляем схожесть между словами
similarity = w2v_model.similarity('computer', 'laptop')
print(f"Схожесть 'computer' и 'laptop': {similarity:.4f}")

Вектор слова 'computer': [-0.16298   0.30141   0.57978   0.066548  0.45835 ]...
Схожесть 'computer' и 'laptop': 0.7024


**Поиск похожих слов**

In [ ]:
# Находим похожие слова
similar_words = w2v_model.most_similar('python', topn=5)
print("Слова, похожие на 'python':")
for word, score in similar_words:
    print(f"  {word}: {score:.4f}")

Слова, похожие на 'python':
  monty: 0.6886
  php: 0.5865
  perl: 0.5784
  cleese: 0.5447
  flipper: 0.5113


*Ваш ответ здесь*

**Задание**

1. Загрузите любой датасет из gensim на ваш выбор

In [8]:
w2v_model_1 = api.load('word2vec-ruscorpora-300')

print(f"Размер словаря: {len(w2v_model.key_to_index)}")
print(f"Размерность векторов: {w2v_model.vector_size}")

Размер словаря: 400000
Размерность векторов: 100


2. Напишите функцию, которая принимает на вход любое слово и вовращает 10 наиболее близких по вектору слов

In [50]:
def similar_words_():
    user_word = input("Введите слово: ").strip().lower()
    try:
        similar_words = w2v_model.most_similar(user_word + '_NOUN', topn=10)
        print(f"Слова, похожие на '{user_word}':")
        for word, score in similar_words:
            print(f"{word}: {score:.4f}")
    except KeyError:
        print(f"Слово '{user_word}' не найдено в словаре модели.")
similar_words_()

Введите слово: лейка
Слова, похожие на 'лейка':
леечка_NOUN: 0.5692
ведро_NOUN: 0.5502
поливать_VERB: 0.5064
ведерко_NOUN: 0.4577
грядка_NOUN: 0.4395
грабли_NOUN: 0.4341
ситечко_NOUN: 0.4306
грабельки_NOUN: 0.4304
кадка_NOUN: 0.4259
пульверизатор_NOUN: 0.4247


3. Обучите модель Word2Vec на тестовом датасете из ячейки ниже

Примените следующие настройки:

- размер вектора: 50
- размер окна: 3
- минимальная частота слова: 1
- потоков: 2
- использовать skip-gram

In [30]:
cooking_sentences = [
    ['варить', 'суп', 'овощи', 'морковь', 'картофель'],
    ['жарить', 'курица', 'сковорода', 'масло', 'специи'],
    ['печь', 'хлеб', 'мука', 'дрожжи', 'духовка'],
    ['резать', 'овощи', 'салат', 'помидоры', 'огурцы'],
    ['смешивать', 'ингредиенты', 'тесто', 'яйца', 'молоко'],
    ['варить', 'паста', 'вода', 'соль', 'соус'],
    ['гриль', 'мясо', 'овощи', 'уголь', 'барбекю'],
    ['тушить', 'говядина', 'горшок', 'вино', 'травы'],
    ['запекать', 'рыба', 'лимон', 'духовка', 'фольга'],
    ['готовить', 'завтрак', 'яичница', 'бекон', 'тост'],
    ['месить', 'тесто', 'пирог', 'начинка', 'яблоки'],
    ['кипятить', 'вода', 'чай', 'кофе', 'чашка'],
    ['мариновать', 'мясо', 'соус', 'специи', 'холодильник'],
    ['взбивать', 'сливки', 'сахар', 'десерт', 'торт'],
    ['парить', 'овощи', 'здоровое', 'питание', 'брокколи']
]

In [49]:
w2v_model_ = Word2Vec(
    sentences=cooking_sentences,
    vector_size=50,
    window=3,
    min_count=1,
    workers=2,
    sg=1
)
def similar_words_():
    user_word = input("Введите слово: ").strip().lower()
    try:
        similar_words = w2v_model_.wv.most_similar(user_word, topn=3)
        print(f"\nСлова, похожие на '{user_word}':")
        for word, score in similar_words:
            print(f"{word}:{score:.4f}")
    except KeyError:
        print(f"Слово '{user_word}' не найдено в словаре модели.")
similar_words_()

Введите слово: питание

Слова, похожие на 'питание':
бекон:0.2637
гриль:0.2545
уголь:0.2347


In [39]:
print(f"Слова в словаре: {list(w2v_model_.wv.key_to_index.keys())[:10]}...")

Слова в словаре: ['овощи', 'мясо', 'соус', 'вода', 'тесто', 'духовка', 'специи', 'варить', 'брокколи', 'питание']...


4. Проверьте модель

In [40]:
# Проверяем похожие слова в кулинарной тематике
try:
    similar = w2v_model_.wv.most_similar('варить', topn=5)
    print("Слова, похожие на 'варить':")
    for word, score in similar:
        print(f"  {word}: {score:.4f}")
except KeyError:
    print("Слово 'варить' не найдено в словаре")

Слова, похожие на 'варить':
  вино: 0.2398
  ингредиенты: 0.2172
  хлеб: 0.1938
  брокколи: 0.1846
  кипятить: 0.1711


In [48]:
# Найдите слова, похожие на "духовка"
### ваш код здесь ###
similar = w2v_model_.wv.most_similar('духовка', topn=5)
print("Слова, похожие на 'духовка':")
for word, score in similar:
    print(f"{word}:{score:.4f}")

# Найдите слова, похожие на "овощи"
### ваш код здесь ###
similar = w2v_model_.wv.most_similar('овощи', topn=5)
print("Слова, похожие на 'овощи':")
for word, score in similar:
    print(f"{word}:{score:.4f}")

Слова, похожие на 'духовка':
ингредиенты:0.3199
десерт:0.3064
холодильник:0.2705
питание:0.2243
пирог:0.2142
Слова, похожие на 'овощи':
мариновать:0.2716
хлеб:0.2691
гриль:0.2546
фольга:0.2409
сахар:0.2108


## Часть 2: FastText

FastText улучшает Word2Vec, рассматривая слова как наборы символов (n-грамм). Это позволяет работать с редкими словами и опечатками

5. Обучите FastText на корпусе текстов из пункта 3. Используйте код ниже

In [52]:
cooking_sentences = [
    ['варить', 'суп', 'овощи', 'морковь', 'картофель'],
    ['жарить', 'курица', 'сковорода', 'масло', 'специи'],
    ['печь', 'хлеб', 'мука', 'дрожжи', 'духовка'],
    ['резать', 'овощи', 'салат', 'помидоры', 'огурцы'],
    ['смешивать', 'ингредиенты', 'тесто', 'яйца', 'молоко'],
    ['варить', 'паста', 'вода', 'соль', 'соус'],
    ['гриль', 'мясо', 'овощи', 'уголь', 'барбекю'],
    ['тушить', 'говядина', 'горшок', 'вино', 'травы'],
    ['запекать', 'рыба', 'лимон', 'духовка', 'фольга'],
    ['готовить', 'завтрак', 'яичница', 'бекон', 'тост'],
    ['месить', 'тесто', 'пирог', 'начинка', 'яблоки'],
    ['кипятить', 'вода', 'чай', 'кофе', 'чашка'],
    ['мариновать', 'мясо', 'соус', 'специи', 'холодильник'],
    ['взбивать', 'сливки', 'сахар', 'десерт', 'торт'],
    ['парить', 'овощи', 'здоровое', 'питание', 'брокколи']
]

ft_model = FastText(
    sentences=cooking_sentences,
    vector_size=50,
    window=3,
    min_count=1,
    workers=2
)

6. Найдите слова, похожие на "варить", "духовка" и "овощи" с помощью обученной модели. Используйте код из пункта 4

In [53]:
similar = ft_model.wv.most_similar('варить', topn=5)
print("Слова, похожие на 'варить':")
for word, score in similar:
    print(f"{word}:{score:.4f}")

similar = ft_model.wv.most_similar('духовка', topn=5)
print("Слова, похожие на 'духовка':")
for word, score in similar:
    print(f"{word}:{score:.4f}")

similar = ft_model.wv.most_similar('овощи', topn=5)
print("Слова, похожие на 'овощи':")
for word, score in similar:
    print(f"{word}:{score:.4f}")

Слова, похожие на 'варить':
жарить:0.5353
парить:0.4805
месить:0.3541
тушить:0.3405
специи:0.2622
Слова, похожие на 'духовка':
взбивать:0.4565
лимон:0.3561
салат:0.3050
курица:0.3041
тост:0.2944
Слова, похожие на 'овощи':
жарить:0.2960
фольга:0.2574
морковь:0.2297
соус:0.2172
торт:0.2094


7. Сравните модели

Дана функция для сравнения Word2Vec и FastText

Придумайте 3 слова с опечатками и проверьте, найдет ли их FastText и Word2Vec

In [55]:
def compare_models(word):
    """Сравнивает представления слова в разных моделях"""
    print(f"\nСравнение для слова: '{word}'")

    # Word2Vec
    try:
        w2v_similar = w2v_model_.wv.most_similar(word, topn=2)
        print(f"  Word2Vec: {[w for w, _ in w2v_similar]}")
    except KeyError:
        print(f"  Word2Vec: слово не найдено")

    # FastText
    try:
        ft_similar = ft_model.wv.most_similar(word, topn=2)
        print(f"  FastText: {[w for w, _ in ft_similar]}")
    except KeyError:
        print(f"  FastText: слово не найдено")

wrong_words = ['даховка', 'марининовать', 'мяс']
for word in wrong_words:
    compare_models(word)

# Сравниваем для разных слов
compare_models('learning')
compare_models('neural')


Сравнение для слова: 'даховка'
  Word2Vec: слово не найдено
  FastText: ['духовка', 'холодильник']

Сравнение для слова: 'марининовать'
  Word2Vec: слово не найдено
  FastText: ['мариновать', 'яичница']

Сравнение для слова: 'мяс'
  Word2Vec: слово не найдено
  FastText: ['мясо', 'питание']

Сравнение для слова: 'learning'
  Word2Vec: слово не найдено
  FastText: ['духовка', 'пирог']

Сравнение для слова: 'neural'
  Word2Vec: слово не найдено
  FastText: ['мука', 'травы']


## Часть 3: Doc2Vec

Doc2Vec расширяет Word2Vec для создания векторных представлений целых документов (предложений, абзацев, статей)

In [56]:
# Создаем размеченные документы
documents = [
    "machine learning is interesting",
    "deep learning uses neural networks",
    "python programming for data science",
    "artificial intelligence is amazing",
    "computer vision processes images"
]

# Преобразуем в формат TaggedDocument
tagged_docs = []
for i, doc in enumerate(documents):
    tokens = doc.split()
    tagged_doc = TaggedDocument(words=tokens, tags=[f"doc_{i}"])
    tagged_docs.append(tagged_doc)

print("Размеченные документы:")
for doc in tagged_docs[:3]:
    print(f"  Слова: {doc.words}")
    print(f"  Тег: {doc.tags}")

Размеченные документы:
  Слова: ['machine', 'learning', 'is', 'interesting']
  Тег: ['doc_0']
  Слова: ['deep', 'learning', 'uses', 'neural', 'networks']
  Тег: ['doc_1']
  Слова: ['python', 'programming', 'for', 'data', 'science']
  Тег: ['doc_2']


In [57]:
# Обучаем Doc2Vec
doc_model = Doc2Vec(
    documents=tagged_docs,
    vector_size=50,
    window=3,
    min_count=1,
    workers=2,
    epochs=20
)

print("Doc2Vec модель обучена!")
print(f"Количество документов: {len(doc_model.dv.key_to_index)}")

Doc2Vec модель обучена!
Количество документов: 5


In [58]:
# Получаем вектор документа
doc_vector = doc_model.dv["doc_0"]
print(f"Вектор документа doc_0: {doc_vector[:5]}...")

# Находим похожие документы
similar_docs = doc_model.dv.most_similar("doc_0", topn=2)
print("\nДокументы, похожие на doc_0:")
for doc_tag, similarity in similar_docs:
    doc_id = int(doc_tag.split('_')[1])
    print(f"  {doc_tag}: {similarity:.4f}")
    print(f"    Текст: {documents[doc_id]}")

Вектор документа doc_0: [-0.01057    -0.01198188 -0.01982618  0.01710627  0.00710373]...

Документы, похожие на doc_0:
  doc_1: 0.2735
    Текст: deep learning uses neural networks
  doc_2: 0.1275
    Текст: python programming for data science


In [59]:
# Сравниваем схожесть документов
def compare_documents(doc1_id, doc2_id):
    similarity = doc_model.dv.similarity(f"doc_{doc1_id}", f"doc_{doc2_id}")
    print(f"Схожесть doc_{doc1_id} и doc_{doc2_id}: {similarity:.4f}")
    print(f"  doc_{doc1_id}: {documents[doc1_id]}")
    print(f"  doc_{doc2_id}: {documents[doc2_id]}")

compare_documents(0, 1)  # machine learning vs deep learning
compare_documents(0, 3)  # machine learning vs AI

Схожесть doc_0 и doc_1: 0.2735
  doc_0: machine learning is interesting
  doc_1: deep learning uses neural networks
Схожесть doc_0 и doc_3: -0.0822
  doc_0: machine learning is interesting
  doc_3: artificial intelligence is amazing


8. Сравните схожесть doc_2 и doc_4

In [60]:
compare_documents(2, 4)

Схожесть doc_2 и doc_4: -0.0362
  doc_2: python programming for data science
  doc_4: computer vision processes images


9. Найдите самый похожий документ на doc_1

In [61]:
a = doc_model.dv.similarity("doc_1", "doc_0")
b = doc_model.dv.similarity("doc_1", "doc_2")
c = doc_model.dv.similarity("doc_1", "doc_3")
d = doc_model.dv.similarity("doc_1", "doc_4")
print(a, b, c, d)

max_value = max(a, b, c, d)

if max_value == a:
    print("Самый похожий на doc_1 — это doc_0")
elif max_value == b:
    print("Самый похожий на doc_1 — это doc_2")
elif max_value == c:
    print("Самый похожий на doc_1 — это doc_3")
elif max_value == d:
    print("Самый похожий на doc_1 — это doc_4")

0.27351698 -0.057311043 0.20308849 -0.25457314
Самый похожий на doc_1 — это doc_0


10. Выберите любую из трёх моделей. Обучите модели с разной размерностью (10, 50, 100). Продемонстрируйте качество их работы на примере поиска похожих слов (выберите любые 3 примера, соответствующих тематике корпуса из пункта 4)

In [62]:
cooking_sentences = [
    ['варить', 'суп', 'овощи', 'морковь', 'картофель'],
    ['жарить', 'курица', 'сковорода', 'масло', 'специи'],
    ['печь', 'хлеб', 'мука', 'дрожжи', 'духовка'],
    ['резать', 'овощи', 'салат', 'помидоры', 'огурцы'],
    ['смешивать', 'ингредиенты', 'тесто', 'яйца', 'молоко'],
    ['варить', 'паста', 'вода', 'соль', 'соус'],
    ['гриль', 'мясо', 'овощи', 'уголь', 'барбекю'],
    ['тушить', 'говядина', 'горшок', 'вино', 'травы'],
    ['запекать', 'рыба', 'лимон', 'духовка', 'фольга'],
    ['готовить', 'завтрак', 'яичница', 'бекон', 'тост'],
    ['месить', 'тесто', 'пирог', 'начинка', 'яблоки'],
    ['кипятить', 'вода', 'чай', 'кофе', 'чашка'],
    ['мариновать', 'мясо', 'соус', 'специи', 'холодильник'],
    ['взбивать', 'сливки', 'сахар', 'десерт', 'торт'],
    ['парить', 'овощи', 'здоровое', 'питание', 'брокколи']
]

ft_model = FastText(
    sentences=cooking_sentences,
    vector_size=10,
    window=3,
    min_count=1,
    workers=2
)

similar = ft_model.wv.most_similar('горшок', topn=5)
print("Слова, похожие на 'горшок':")
for word, score in similar:
    print(f"{word}:{score:.4f}")

similar = ft_model.wv.most_similar('чашка', topn=5)
print("Слова, похожие на 'чашка':")
for word, score in similar:
    print(f"{word}:{score:.4f}")

similar = ft_model.wv.most_similar('начинка', topn=5)
print("Слова, похожие на 'начинка':")
for word, score in similar:
    print(f"{word}:{score:.4f}")

Слова, похожие на 'горшок':
ингредиенты:0.8008
вода:0.5224
чай:0.5081
огурцы:0.4896
молоко:0.4483
Слова, похожие на 'чашка':
травы:0.8208
огурцы:0.6147
кофе:0.5918
питание:0.5894
печь:0.5739
Слова, похожие на 'начинка':
мариновать:0.6388
сахар:0.4972
тушить:0.4572
лимон:0.3899
завтрак:0.3890


In [63]:
ft_model = FastText(
    sentences=cooking_sentences,
    vector_size=50,
    window=3,
    min_count=1,
    workers=2
)

similar = ft_model.wv.most_similar('горшок', topn=5)
print("Слова, похожие на 'горшок':")
for word, score in similar:
    print(f"{word}:{score:.4f}")

similar = ft_model.wv.most_similar('чашка', topn=5)
print("Слова, похожие на 'чашка':")
for word, score in similar:
    print(f"{word}:{score:.4f}")

similar = ft_model.wv.most_similar('начинка', topn=5)
print("Слова, похожие на 'начинка':")
for word, score in similar:
    print(f"{word}:{score:.4f}")

Слова, похожие на 'горшок':
чашка:0.2603
гриль:0.2296
яичница:0.1915
тесто:0.1913
сковорода:0.1802
Слова, похожие на 'чашка':
мука:0.2984
горшок:0.2603
молоко:0.1997
месить:0.1778
специи:0.1579
Слова, похожие на 'начинка':
уголь:0.2947
молоко:0.2786
резать:0.2748
смешивать:0.2508
чай:0.2298


In [64]:
ft_model = FastText(
    sentences=cooking_sentences,
    vector_size=100,
    window=3,
    min_count=1,
    workers=2
)

similar = ft_model.wv.most_similar('горшок', topn=5)
print("Слова, похожие на 'горшок':")
for word, score in similar:
    print(f"{word}:{score:.4f}")

similar = ft_model.wv.most_similar('чашка', topn=5)
print("Слова, похожие на 'чашка':")
for word, score in similar:
    print(f"{word}:{score:.4f}")

similar = ft_model.wv.most_similar('начинка', topn=5)
print("Слова, похожие на 'начинка':")
for word, score in similar:
    print(f"{word}:{score:.4f}")

Слова, похожие на 'горшок':
здоровое:0.2399
мариновать:0.2179
тесто:0.2045
специи:0.1882
десерт:0.1837
Слова, похожие на 'чашка':
мариновать:0.2151
готовить:0.2097
салат:0.1928
ингредиенты:0.1681
яичница:0.1627
Слова, похожие на 'начинка':
яблоки:0.3366
сливки:0.3007
морковь:0.2461
тост:0.1723
сковорода:0.1596
